In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")

In [2]:
BASE_DIR = Path().resolve().parent
DATA_PATH = BASE_DIR / "data" / "email_access_data.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,text,label
0,Access granted to Analytics Server.,approved
1,Manager approval not received for HR Portal.,rejected
2,Unable to process your request for Finance Das...,manual
3,Request could not be understood.,manual
4,Request could not be understood.,manual


In [3]:
label_map = {
    "approved": 0,
    "rejected": 1,
    "manual": 2
}

df["label_id"] = df["label"].map(label_map)

df.head()

,text,label,label_id
0,Access granted to Analytics Server.,approved,0
1,Manager approval not received for HR Portal.,rejected,1
2,Unable to process your request for Finance Das...,manual,2
3,Request could not be understood.,manual,2
4,Request could not be understood.,manual,2


In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label_id"],
    random_state=42
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain distribution:")
print(train_df["label"].value_counts())

print("\nTest distribution:")
print(test_df["label"].value_counts())

Train shape: (398, 3)
Test shape: (100, 3)

Train distribution:
label
approved    133
rejected    133
manual      132
Name: count, dtype: int64

Test distribution:
label
manual      34
rejected    33
approved    33
Name: count, dtype: int64


In [5]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

c:\Users\enaarmp\OneDrive - Ericsson\My Data\My Learning Track\Email_classifier\Manoj_email_classifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
sample_text = train_df["text"].iloc[0]

tokens = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

tokens

{'input_ids': tensor([[  101,  2115,  3229,  5227,  2000, 13675,  2213,  2291,  2038,  2042,
          4844,  1012,   102,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [7]:
import torch
from torch.utils.data import Dataset

class EmailDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=64
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [8]:
train_dataset = EmailDataset(
    train_df["text"].tolist(),
    train_df["label_id"].tolist()
)

test_dataset = EmailDataset(
    test_df["text"].tolist(),
    test_df["label_id"].tolist()
)

len(train_dataset), len(test_dataset)

(398, 100)

In [9]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 280.50it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

In [13]:
from transformers import Trainer,TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    # "evaluation_strategy"="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    # load_best_model_at_end=True,
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [14]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted"
    )
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

c:\Users\enaarmp\OneDrive - Ericsson\My Data\My Learning Track\Email_classifier\Manoj_email_classifier\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.99s/it]
c:\Users\enaarmp\OneDrive - Ericsson\My Data\My Learning Track\Email_classifier\Manoj_email_classifier\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.98s/it]
c:\Users\enaarmp\OneDrive - Ericsson\My Data\My Learning Track\Email_classifier\Manoj_email_classifier\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.40s/it]


TrainOutput(global_step=75, training_loss=0.18120223999023438, metrics={'train_runtime': 163.7894, 'train_samples_per_second': 7.29, 'train_steps_per_second': 0.458, 'total_flos': 7976653261668.0, 'train_loss': 0.18120223999023438, 'epoch': 3.0})

In [16]:
trainer.evaluate()

c:\Users\enaarmp\OneDrive - Ericsson\My Data\My Learning Track\Email_classifier\Manoj_email_classifier\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.0037279371172189713,
 'eval_accuracy': 1.0,
 'eval_f1': 1.0,
 'eval_precision': 1.0,
 'eval_recall': 1.0,
 'eval_runtime': 1.1276,
 'eval_samples_per_second': 88.682,
 'eval_steps_per_second': 6.208,
 'epoch': 3.0}

In [17]:
model.save_pretrained("../models/email_classifier")
tokenizer.save_pretrained("../models/email_classifier")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]


('../models/email_classifier\\tokenizer_config.json',
 '../models/email_classifier\\tokenizer.json')

Inferance

In [18]:
import torch
import torch.nn.functional as F

id2label = {0: "approved", 1: "rejected", 2: "manual"}

def predict_email(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = F.softmax(outputs.logits, dim=1)
    confidence, pred_class = torch.max(probs, dim=1)
    
    return {
        "prediction": id2label[pred_class.item()],
        "confidence": round(confidence.item(), 4)
    }

In [20]:
predict_email("Your access to VPN has been approved.")

{'prediction': 'approved', 'confidence': 0.9968}

In [21]:
predict_email("Manager approval not received for Finance Dashboard.")

{'prediction': 'rejected', 'confidence': 0.9967}

In [22]:
predict_email("Please contact admin for further processing.")

{'prediction': 'manual', 'confidence': 0.9942}